## Importing files & Scraping pdf

In [1]:
import pandas as pd
import camelot

pdf_path = "C:/Users/user/OneDrive - Hetauda School of Management and Social Sciences/Desktop/Data Analysis/Projects/Project V/Raw Datas/SUN2000-200KTL-H2-Datasheet-20230822.pdf"
tables = camelot.read_pdf(pdf_path, flavor = 'stream', pages = '2', edge_tol = 100);

data = tables[0].df
data

,0,1
0,,Efficiency
1,Max. Efficiency,≥99.00%
2,European Efficiency,≥98.80%
3,,Input
4,Max. Input Voltage,"1,500 V"
5,Max. Current per MPPT,30 A
6,Max. Short Circuit Current per MPPT,50 A
7,Start Voltage,550 V
8,MPPT Operating Voltage Range,"500 V ~ 1,500 V"
9,Nominal Input Voltage,"1,080 V"


## Data Cleaning

In [12]:
target_params = [
    "Max. Efficiency", "European Efficiency", "Max. Input Voltage", 
    "MPPT Operating Voltage Range", "Max. Current per MPPT", 
    "AC Output Power", "Max. AC Apparent Power", "Nominal Output Voltage", 
    "Max. Output Current", "Number of MPP Trackers"
]

df_clean = data[data[0].isin(target_params)].copy();


df_clean.loc[:, 1] = df_clean[1].str.replace(',', ''); 


df_clean.loc[:, 1] = df_clean[1].str.extract(r'(\d[\d\s\.\~]*)')[0].str.strip()
# \d: Start with exactly one digit (0-9). This ensures we don't grab symbols like ≥ or <.
# [ ... ]*: This is a "character class" that says: "After that first digit, keep grabbing anything inside these brackets until you hit something else."
# \d: More digits.
# \s: Spaces (needed for ranges like 500 ~ 1500).
# \.: Decimal points (for values like 99.8).
# \~: The tilde symbol (to keep ranges intact).

df_clean.loc[:, 1] = df_clean[1].str.replace('~', '-')

category_map = {
    "Max. Efficiency": "Performance", "European Efficiency": "Performance",
    "Max. Input Voltage": "Input", "MPPT Operating Voltage Range": "Input",
    "Max. Current per MPPT": "Input", "AC Output Power": "Output",
    "Max. AC Apparent Power": "Output", "Nominal Output Voltage": "Output",
    "Max. Output Current": "Output", "Number of MPP Trackers": "Design"
}
df_clean.loc[:, 'Category'] = df_clean[0].map(category_map)


df_clean.columns = ['Parameter', 'Value', 'Category']

eff_mask = df_clean['Parameter'].str.contains('Efficiency')
df_clean.loc[eff_mask, 'Value'] = df_clean.loc[eff_mask, 'Value'] + '%'

df_clean

,Parameter,Value,Category
1,Max. Efficiency,99.00%,Performance
2,European Efficiency,98.80%,Performance
4,Max. Input Voltage,1500,Input
5,Max. Current per MPPT,30,Input
8,MPPT Operating Voltage Range,500,Input
11,Number of MPP Trackers,9,Design
13,AC Output Power,200000,Output
14,Max. AC Apparent Power,215000,Output
16,Nominal Output Voltage,800,Output
19,Max. Output Current,155.2,Output


## Exporting